In [ ]:
# imports
import os
from dotenv import load_dotenv
from time import perf_counter

from google import genai

from pydantic import BaseModel, Field
from typing import List, Optional

In [ ]:
# loading api key
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
# creating the client
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# pydantic model for an Ingredient
class Ingredient(BaseModel):
    name: str = Field(description="Name of the ingredient.")
    quantity: str = Field(description="Quantity of the ingredient, including units.")

# pydantic model for a Recipy
class Recipe(BaseModel):
    recipe_name: str = Field(description="The name of the recipe.")
    prep_time_minutes: Optional[int] = Field(description="Optional time in minutes to prepare the recipe.")
    ingredients: List[Ingredient]
    instructions: List[str]

In [ ]:
# standard gemini prompt from docs
prompt = """
Please extract the recipe from the following text.
The user wants to make delicious chocolate chip cookies.
They need 2 and 1/4 cups of all-purpose flour, 1 teaspoon of baking soda,
1 teaspoon of salt, 1 cup of unsalted butter (softened), 3/4 cup of granulated sugar,
3/4 cup of packed brown sugar, 1 teaspoon of vanilla extract, and 2 large eggs.
For the best part, they'll need 2 cups of semisweet chocolate chips.
First, preheat the oven to 375°F (190°C). Then, in a small bowl, whisk together the flour,
baking soda, and salt. In a large bowl, cream together the butter, granulated sugar, and brown sugar
until light and fluffy. Beat in the vanilla and eggs, one at a time. Gradually beat in the dry
ingredients until just combined. Finally, stir in the chocolate chips. Drop by rounded tablespoons
onto ungreased baking sheets and bake for 9 to 11 minutes.
"""

In [ ]:
# making a call and measuring the time to process it
start_time = perf_counter()

interaction = client.interactions.create(
    model="gemini-3.5-flash-lite",
    input=prompt,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema_": Recipe.model_json_schema()
    },
)

end_time = perf_counter()

total_time = (end_time - start_time)

In [ ]:
# validating the output
recipe = Recipe.model_validate_json(interaction.output_text)

In [ ]:
# printing validated output
print(recipe)
print(type(recipe))

In [ ]:
# printing plain output
print(interaction.output_text)
print(type(interaction.output_text))

In [ ]:
# used token count
input_tokens_count = interaction.usage.total_input_tokens
output_tokens_count = interaction.usage.total_output_tokens

total_tokens = input_tokens_count + output_tokens_count

In [ ]:
# creating statistics object
statistics = {
    "time": total_time,
    "token_count": total_tokens
}

In [ ]:
print(statistics)